In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/df_fase1.csv')
print('Dataset carregado de data/df_fase1.csv')


Dataset carregado de data/df_fase1.csv


---
# Fase 3 — Pré-processamento

**Objetivo:** Preparar os dados para a modelagem, tratando valores ausentes, codificando variáveis categóricas, transformando a variável-alvo e separando features e alvo.

> ⚠️ **Regra fundamental sobre Data Leakage:** Qualquer transformação que utilize informações estatísticas do dataset (média, mediana, percentis, desvio-padrão) deve ser ajustada (`.fit()`) **exclusivamente nos dados de treinamento**. Os dados de teste devem ser transformados (`.transform()`) utilizando os parâmetros aprendidos no treino. Isso será garantido pela utilização de **Pipelines** na Fase 5.

## 3.1 — Transformação da Variável-Alvo

A variável `Dataset` originalmente utiliza os valores:
- `1` = doença hepática
- `2` = sem doença hepática

Será convertida para classificação binária padrão:
- `1` = doença hepática (classe positiva)
- `0` = sem doença hepática (classe negativa)

Essa transformação é realizada **antes** da separação entre features e alvo.

In [2]:
# Verificar estado atual da variável-alvo
print('Antes da transformação:')
print(df['Dataset'].value_counts().sort_index())

# Transformar: 1 -> 1 (doença), 2 -> 0 (sem doença)
df['Dataset'] = df['Dataset'].map({1: 1, 2: 0})

print('\nApós a transformação:')
print(df['Dataset'].value_counts().sort_index())

# Validação
assert set(df['Dataset'].unique()) == {0, 1}, 'Erro: valores inesperados na variável-alvo!'
assert df['Dataset'].sum() == 416, 'Erro: contagem da classe positiva não confere!'
print('\n\u2713 Variável-alvo transformada com sucesso: 1 = doença hepática, 0 = sem doença.')

Antes da transformação:
Dataset
1    416
2    167
Name: count, dtype: int64

Após a transformação:
Dataset
0    167
1    416
Name: count, dtype: int64

✓ Variável-alvo transformada com sucesso: 1 = doença hepática, 0 = sem doença.


## 3.2 — Codificação de `Gender`

A variável `Gender` é categórica binária com dois valores: `Male` e `Female`.

**Estratégia:** Mapeamento direto:
- `Male` → `1`
- `Female` → `0`

**Justificativa:** Por se tratar de uma variável binária, a codificação simples é suficiente e produz o mesmo resultado que One-Hot Encoding com `drop='first'`. Não há ordinalidade artificial neste caso.

In [3]:
# Verificar estado atual
print('Antes da codificação:')
print(df['Gender'].value_counts())

# Codificar: Male -> 1, Female -> 0
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

print('\nApós a codificação:')
print(df['Gender'].value_counts().sort_index())

# Validação
assert set(df['Gender'].unique()) == {0, 1}, 'Erro: valores inesperados em Gender!'
assert df['Gender'].dtype in ['int64', 'int32', 'float64'], 'Erro: tipo inesperado em Gender!'
print('\n\u2713 Gender codificado com sucesso: 1 = Male, 0 = Female.')

Antes da codificação:
Gender
Male      441
Female    142
Name: count, dtype: int64

Após a codificação:
Gender
0    142
1    441
Name: count, dtype: int64

✓ Gender codificado com sucesso: 1 = Male, 0 = Female.


## 3.3 — Tratamento dos Valores Ausentes

A coluna `Albumin_and_Globulin_Ratio` possui **4 valores ausentes** (0,69% do dataset).

**Estratégia planejada:** Imputação pela **mediana**, calculada exclusivamente a partir dos dados de treinamento.

**Justificativa:** A mediana é robusta a outliers — característica importante dado que a EDA identificou outliers nas features clínicas.

> ⚠️ **Nesta fase, apenas documentamos a estratégia.** A imputação efetiva será realizada dentro do **Pipeline** (Fase 5), garantindo que o valor da mediana seja calculado **somente com os dados de treinamento** e aplicado ao teste via `.transform()`. Isso evita data leakage.

Abaixo, verificamos o estado atual dos valores ausentes para referência.

In [4]:
# Estado atual dos valores ausentes
ausentes = df.isnull().sum()
print('Valores ausentes por coluna:')
print(ausentes[ausentes > 0])
print(f'\nTotal de registros com valores ausentes: {df.isnull().any(axis=1).sum()}')

# Mediana atual (apenas para referência — NÃO será usada para imputação)
mediana_ag = df['Albumin_and_Globulin_Ratio'].median()
print(f'\nMediana atual de Albumin_and_Globulin_Ratio (dataset completo): {mediana_ag:.3f}')
print('  Nota: este valor é apenas referencial. A imputação será feita no Pipeline')
print('  usando a mediana calculada SOMENTE nos dados de treinamento.')

Valores ausentes por coluna:
Albumin_and_Globulin_Ratio    4
dtype: int64

Total de registros com valores ausentes: 4

Mediana atual de Albumin_and_Globulin_Ratio (dataset completo): 0.930
  Nota: este valor é apenas referencial. A imputação será feita no Pipeline
  usando a mediana calculada SOMENTE nos dados de treinamento.


## 3.4 — Separação entre Features (X) e Alvo (y)

Após as transformações da variável-alvo e da codificação de `Gender`, separamos o DataFrame em:
- `X` — matriz de features preditoras (10 colunas)
- `y` — vetor da variável-alvo (1 = doença, 0 = sem doença)

In [5]:
# Separação features / alvo
X = df.drop('Dataset', axis=1)
y = df['Dataset']

print(f'X (features):  {X.shape[0]} registros \u00d7 {X.shape[1]} features')
print(f'y (alvo):      {y.shape[0]} registros')
print(f'\nFeatures: {list(X.columns)}')
print(f'\nDistribui\u00e7\u00e3o do alvo:')
print(f'  1 (doen\u00e7a):      {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)')
print(f'  0 (sem doen\u00e7a):  {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)')

X (features):  583 registros × 10 features
y (alvo):      583 registros

Features: ['Age', 'Gender', 'Total_Bilirubin', 'Direct_Bilirubin', 'Alkaline_Phosphotase', 'Alamine_Aminotransferase', 'Aspartate_Aminotransferase', 'Total_Protiens', 'Albumin', 'Albumin_and_Globulin_Ratio']

Distribuição do alvo:
  1 (doença):      416 (71.4%)
  0 (sem doença):  167 (28.6%)


In [6]:
# Verificação do estado final do DataFrame de features
print('Tipos de dados em X:')
print(X.dtypes)
print(f'\nValores ausentes em X:')
print(X.isnull().sum()[X.isnull().sum() > 0])
print(f'\nPrimeiras linhas de X:')
X.head()

Tipos de dados em X:
Age                             int64
Gender                          int64
Total_Bilirubin               float64
Direct_Bilirubin              float64
Alkaline_Phosphotase            int64
Alamine_Aminotransferase        int64
Aspartate_Aminotransferase      int64
Total_Protiens                float64
Albumin                       float64
Albumin_and_Globulin_Ratio    float64
dtype: object

Valores ausentes em X:
Albumin_and_Globulin_Ratio    4
dtype: int64

Primeiras linhas de X:


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio
0,65,0,0.7,0.1,187,16,18,6.8,3.3,0.90
1,62,1,10.9,5.5,699,64,100,7.5,3.2,0.74
2,62,1,7.3,4.1,490,60,68,7.0,3.3,0.89
3,58,1,1.0,0.4,182,14,20,6.8,3.4,1.00
4,72,1,3.9,2.0,195,27,59,7.3,2.4,0.40


## 3.5 — Identificação de Problemas de Escala

As features numéricas possuem escalas muito diferentes, o que impacta modelos sensíveis à distância e à regularização.

In [7]:
# Comparar escalas das features
escala = pd.DataFrame({
    'Min': X.min(),
    'Max': X.max(),
    'M\u00e9dia': X.mean(),
    'Desvio Padr\u00e3o': X.std(),
    'Amplitude': X.max() - X.min()
}).round(2)

print('Compara\u00e7\u00e3o de escalas das features:')
escala

Comparação de escalas das features:


,Min,Max,Média,Desvio Padrão,Amplitude
Age,4.0,90.0,44.75,16.19,86.0
Gender,0.0,1.0,0.76,0.43,1.0
Total_Bilirubin,0.4,75.0,3.30,6.21,74.6
Direct_Bilirubin,0.1,19.7,1.49,2.81,19.6
Alkaline_Phosphotase,63.0,2110.0,290.58,242.94,2047.0
Alamine_Aminotransferase,10.0,2000.0,80.71,182.62,1990.0
Aspartate_Aminotransferase,10.0,4929.0,109.91,288.92,4919.0
Total_Protiens,2.7,9.6,6.48,1.09,6.9
Albumin,0.9,5.5,3.14,0.80,4.6
Albumin_and_Globulin_Ratio,0.3,2.8,0.95,0.32,2.5


**Observações sobre escala:**

As amplitudes variam enormemente:
- `Gender`: 0–1 (binária)
- `Age`: 4–90
- `Alkaline_Phosphotase`: 63–2110
- `Aspartate_Aminotransferase`: 10–4929

**Impacto:**
| Modelo | Sensível à escala? | Ação |
|---|---|---|
| Regressão Logística | Sim | Padronizar (StandardScaler) |
| KNN | Sim | Padronizar |
| SVM | Sim | Padronizar |
| Random Forest | Não | Pode omitir |
| XGBoost | Não | Pode omitir |

A padronização será implementada dentro do Pipeline (Fase 5).

## 3.6 — Tratamento de Outliers

Com base nas descobertas da EDA (Fase 2):

In [8]:
# Resumo dos outliers identificados na EDA
features_numericas = ['Age', 'Total_Bilirubin', 'Direct_Bilirubin',
                      'Alkaline_Phosphotase', 'Alamine_Aminotransferase',
                      'Aspartate_Aminotransferase', 'Total_Protiens',
                      'Albumin', 'Albumin_and_Globulin_Ratio']

print('Contagem de outliers por feature (m\u00e9todo IQR):')
print('-' * 50)
for col in features_numericas:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((X[col] < Q1 - 1.5 * IQR) | (X[col] > Q3 + 1.5 * IQR)).sum()
    print(f'  {col:35s} -> {n_out:3d} outliers')
print('-' * 50)

Contagem de outliers por feature (método IQR):
--------------------------------------------------
  Age                                 ->   0 outliers
  Total_Bilirubin                     ->  84 outliers
  Direct_Bilirubin                    ->  81 outliers
  Alkaline_Phosphotase                ->  69 outliers
  Alamine_Aminotransferase            ->  73 outliers
  Aspartate_Aminotransferase          ->  66 outliers
  Total_Protiens                      ->   8 outliers
  Albumin                             ->   0 outliers
  Albumin_and_Globulin_Ratio          ->  10 outliers
--------------------------------------------------


### Decisão sobre outliers

**Decisão: Manter os outliers.** Justificativa:

1. **Plausibilidade clínica:** Os valores extremos de bilirrubina e enzimas hepáticas são clinicamente possíveis em pacientes com doença hepática grave (hepatite aguda, cirrose, obstrução biliar).

2. **Informação diagnóstica:** Valores extremos podem ser justamente os mais informativos para a classificação — removê-los poderia reduzir o poder preditivo do modelo.

3. **Mitigação via modelagem:**
   - `StandardScaler` reduz o impacto de outliers em modelos lineares.
   - Modelos baseados em árvore (Random Forest, XGBoost) são naturalmente robustos a outliers.
   - A regularização (L1/L2) limita a influência de coeficientes extremos.

4. **Tamanho do dataset:** Com apenas 583 registros, a remoção de outliers reduziria significativamente o volume de dados disponíveis para treinamento.

> Se, após a modelagem, houver evidência de que outliers estão prejudicando o desempenho, poderão ser aplicadas transformações (ex.: `log1p`) ou capping. Qualquer transformação desse tipo deverá ser ajustada **somente nos dados de treinamento**.

## 3.7 — Resumo do Pré-processamento

| Etapa | Ação | Status |
|---|---|---|
| Variável-alvo | `Dataset`: {1,2} → {1,0} | ✅ Concluído |
| Codificação | `Gender`: Male/Female → 1/0 | ✅ Concluído |
| Valores ausentes | 4 em `Albumin_and_Globulin_Ratio` | ⏳ Será tratado no Pipeline (mediana do treino) |
| Separação X/y | 10 features + 1 alvo | ✅ Concluído |
| Escala | Amplitudes muito diferentes entre features | ⏳ Será tratado no Pipeline (StandardScaler) |
| Outliers | Presentes em bilirrubina e enzimas | ✅ Decisão: Manter |
| Data leakage | Transformações estatísticas adiadas para Pipeline | ✅ Garantido |

In [9]:
# Estado final após pré-processamento
print('=== Estado Final do Pré-processamento ===')
print(f'\nX shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'\nTipos de dados em X:')
print(X.dtypes.to_string())
print(f'\nValores ausentes restantes: {X.isnull().sum().sum()}')
print(f'  (ser\u00e3o tratados no Pipeline da Fase 5)')
print(f'\ny dtype: {y.dtype}')
print(f'y distribui\u00e7\u00e3o: 1={int((y==1).sum())}, 0={int((y==0).sum())}')
print(f'\n\u2713 Dados prontos para a Fase 4 (Divis\u00e3o dos Dados).')

=== Estado Final do Pré-processamento ===

X shape: (583, 10)
y shape: (583,)

Tipos de dados em X:
Age                             int64
Gender                          int64
Total_Bilirubin               float64
Direct_Bilirubin              float64
Alkaline_Phosphotase            int64
Alamine_Aminotransferase        int64
Aspartate_Aminotransferase      int64
Total_Protiens                float64
Albumin                       float64
Albumin_and_Globulin_Ratio    float64

Valores ausentes restantes: 4
  (serão tratados no Pipeline da Fase 5)

y dtype: int64
y distribuição: 1=416, 0=167

✓ Dados prontos para a Fase 4 (Divisão dos Dados).


### Checklist de Conclusão da Fase 3

- ✅ Variável-alvo transformada para formato binário (1/0).
- ✅ Variável `Gender` codificada numericamente (Male=1, Female=0).
- ✅ Valores ausentes documentados — imputação será feita no Pipeline.
- ✅ Features e alvo separados em `X` e `y`.
- ✅ Problemas de escala identificados e estratégia definida.
- ✅ Decisão sobre outliers documentada e justificada (manter).
- ✅ Nenhuma transformação baseada em estatísticas do dataset completo (sem data leakage).

In [10]:
X.to_csv('data/X_fase3.csv', index=False)
y.to_csv('data/y_fase3.csv', index=False)
print('X e y salvos em data/')


X e y salvos em data/
